In [1]:

from pathlib import Path
from zipfile import ZipFile
import pandas as pd

zip_path = Path("/content/incident+management+process+enriched+event+log.zip")

with ZipFile(zip_path, "r") as z:
    print("Files inside ZIP:", z.namelist())
    z.extractall("/content/sla_data")

# Find the CSV file
csv_files = list(Path("/content/sla_data").rglob("*.csv"))
print("CSV files found:", csv_files)

assert csv_files, "No CSV file found in ZIP"

# Load data
df = pd.read_csv(
    csv_files[0],
    dtype=str,
    keep_default_na=False,
    low_memory=False
)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())


Files inside ZIP: ['incident_event_log.csv']
CSV files found: [PosixPath('/content/sla_data/incident_event_log.csv')]
Dataset shape: (141712, 36)
Columns: ['number', 'incident_state', 'active', 'reassignment_count', 'reopen_count', 'sys_mod_count', 'made_sla', 'caller_id', 'opened_by', 'opened_at', 'sys_created_by', 'sys_created_at', 'sys_updated_by', 'sys_updated_at', 'contact_type', 'location', 'category', 'subcategory', 'u_symptom', 'cmdb_ci', 'impact', 'urgency', 'priority', 'assignment_group', 'assigned_to', 'knowledge', 'u_priority_confirmation', 'notify', 'problem_id', 'rfc', 'vendor', 'caused_by', 'closed_code', 'resolved_by', 'resolved_at', 'closed_at']


,number,incident_state,active,reassignment_count,reopen_count,sys_mod_count,made_sla,caller_id,opened_by,opened_at,...,u_priority_confirmation,notify,problem_id,rfc,vendor,caused_by,closed_code,resolved_by,resolved_at,closed_at
0,INC0000045,New,true,0,0,0,true,Caller 2403,Opened by 8,29/2/2016 01:16,...,false,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
1,INC0000045,Resolved,true,0,0,2,true,Caller 2403,Opened by 8,29/2/2016 01:16,...,false,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
2,INC0000045,Resolved,true,0,0,3,true,Caller 2403,Opened by 8,29/2/2016 01:16,...,false,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
3,INC0000045,Closed,false,0,0,4,true,Caller 2403,Opened by 8,29/2/2016 01:16,...,false,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
4,INC0000047,New,true,0,0,0,true,Caller 2403,Opened by 397,29/2/2016 04:40,...,false,Do Not Notify,?,?,?,?,code 5,Resolved by 81,1/3/2016 09:52,6/3/2016 10:00


In [2]:

required_columns = [
    "number",
    "opened_at",
    "sys_updated_at",
    "priority",
    "incident_state",
    "resolved_at"
]

print("Missing columns:")
print(set(required_columns) - set(df.columns))

print("\nUnique incidents:")
print(df["number"].nunique())

print("\nPriority distribution:")
print(df["priority"].value_counts(dropna=False))

print("\nIncident states:")
print(df["incident_state"].value_counts(dropna=False))

print("\nExact duplicate rows:")
print(df.duplicated().sum())


Missing columns:
set()

Unique incidents:
24918

Priority distribution:
priority
3 - Moderate    132452
4 - Low           4030
2 - High          2972
1 - Critical      2258
Name: count, dtype: int64

Incident states:
incident_state
Active                38716
New                   36407
Resolved              25751
Closed                24985
Awaiting User Info    14642
Awaiting Vendor         707
Awaiting Problem        461
Awaiting Evidence        38
-100                      5
Name: count, dtype: int64

Exact duplicate rows:
0


In [3]:

import pandas as pd

timestamp_columns = [
    "opened_at",
    "sys_created_at",
    "sys_updated_at",
    "resolved_at",
    "closed_at"
]

missing_tokens = ["", "?", "NULL", "null", "None"]

for col in timestamp_columns:
    values = df[col].replace(missing_tokens, pd.NA)

    parsed = pd.to_datetime(
        values,
        format="%d/%m/%Y %H:%M",
        errors="coerce"
    )

    df[col + "_parsed"] = parsed

    print(f"\n--- {col} ---")
    print("Missing:", values.isna().sum())
    print("Parse failures:",
          (values.notna() & parsed.isna()).sum())
    print("Earliest:", parsed.min())
    print("Latest:", parsed.max())

print("\nInvalid chronological relationships:")

valid_open_update = (
    df["opened_at_parsed"].notna()
    & df["sys_updated_at_parsed"].notna()
)

print(
    "Updates before opening:",
    (
        valid_open_update
        & (
            df["sys_updated_at_parsed"]
            < df["opened_at_parsed"]
        )
    ).sum()
)

valid_open_resolved = (
    df["opened_at_parsed"].notna()
    & df["resolved_at_parsed"].notna()
)

print(
    "Resolution before opening:",
    (
        valid_open_resolved
        & (
            df["resolved_at_parsed"]
            < df["opened_at_parsed"]
        )
    ).sum()
)



--- opened_at ---
Missing: 0
Parse failures: 0
Earliest: 2016-02-29 01:16:00
Latest: 2017-02-16 14:17:00

--- sys_created_at ---
Missing: 53076
Parse failures: 0
Earliest: 2016-02-29 01:23:00
Latest: 2017-01-27 16:59:00

--- sys_updated_at ---
Missing: 0
Parse failures: 0
Earliest: 2016-02-29 01:23:00
Latest: 2017-02-18 15:00:00

--- resolved_at ---
Missing: 3141
Parse failures: 0
Earliest: 2016-02-29 09:04:00
Latest: 2017-02-17 00:47:00

--- closed_at ---
Missing: 0
Parse failures: 0
Earliest: 2016-02-29 17:47:00
Latest: 2017-02-18 15:00:00

Invalid chronological relationships:
Updates before opening: 5
Resolution before opening: 0


In [5]:

valid = (
    df["resolved_at_parsed"].notna()
    & df["sys_updated_at_parsed"].notna()
)

future_resolution = (
    valid
    & (
        df["resolved_at_parsed"]
        > df["sys_updated_at_parsed"]
    )
)

print("Future resolution timestamps in event rows:",
      int(future_resolution.sum()))

print("Unique incidents affected:",
      df.loc[future_resolution, "number"].nunique())

print("\nExample affected rows:")
display(
    df.loc[
        future_resolution,
        [
            "number",
            "incident_state",
            "sys_updated_at",
            "resolved_at"
        ]
    ].head(10)
)

print("\nEvents per incident:")
print(df.groupby("number").size().describe())

print("\nReopen count distribution:")
print(df["reopen_count"].value_counts().head(15))

print("\n5 events updated before opening:")
invalid_updates = (
    df["sys_updated_at_parsed"]
    < df["opened_at_parsed"]
)

display(
    df.loc[
        invalid_updates,
        ["number", "opened_at", "sys_updated_at",
         "incident_state"]
    ]
)


Future resolution timestamps in event rows: 87471
Unique incidents affected: 22894

Example affected rows:


,number,incident_state,sys_updated_at,resolved_at
0,INC0000045,New,29/2/2016 01:23,29/2/2016 11:29
1,INC0000045,Resolved,29/2/2016 08:53,29/2/2016 11:29
4,INC0000047,New,29/2/2016 04:57,1/3/2016 09:52
5,INC0000047,Active,29/2/2016 05:30,1/3/2016 09:52
6,INC0000047,Active,29/2/2016 05:33,1/3/2016 09:52
7,INC0000047,Active,29/2/2016 11:31,1/3/2016 09:52
8,INC0000047,Active,29/2/2016 11:32,1/3/2016 09:52
9,INC0000047,Active,1/3/2016 09:14,1/3/2016 09:52
10,INC0000047,Awaiting User Info,1/3/2016 09:14,1/3/2016 09:52
13,INC0000057,New,29/2/2016 06:26,1/3/2016 02:55



Events per incident:
count    24918.000000
mean         5.687134
std          3.677845
min          2.000000
25%          3.000000
50%          5.000000
75%          7.000000
max         58.000000
dtype: float64

Reopen count distribution:
reopen_count
0    139398
1      1918
2       190
3       107
4        48
5        24
6        18
7         5
8         4
Name: count, dtype: int64

5 events updated before opening:


,number,opened_at,sys_updated_at,incident_state
328,INC0000130,29/2/2016 09:41,29/2/2016 07:27,New
334,INC0000131,29/2/2016 09:41,29/2/2016 07:14,Active
344,INC0000134,29/2/2016 09:44,29/2/2016 07:55,New
349,INC0000135,29/2/2016 09:45,29/2/2016 09:27,Active
436,INC0000164,29/2/2016 10:18,29/2/2016 08:53,New


In [6]:

import pandas as pd

# Treat missing values consistently
missing_tokens = ["", "?", "NULL", "null", "None"]

audit_df = df.copy()

for col in ["opened_at", "resolved_at", "priority",
            "category", "reopen_count"]:
    audit_df[col] = audit_df[col].replace(
        missing_tokens, pd.NA
    )

grouped = audit_df.groupby("number")

summary = pd.DataFrame({
    "event_count": grouped.size(),
    "opened_at_versions": grouped["opened_at"].nunique(),
    "resolved_at_versions": grouped["resolved_at"].nunique(),
    "priority_versions": grouped["priority"].nunique(),
    "category_versions": grouped["category"].nunique(),
})

reopens = pd.to_numeric(
    audit_df["reopen_count"], errors="coerce"
)

summary["max_reopen_count"] = (
    reopens.groupby(audit_df["number"]).max()
)

print("Total unique incidents:", len(summary))

print("\nIncidents with multiple opening timestamps:")
print((summary["opened_at_versions"] > 1).sum())

print("\nIncidents with multiple resolution timestamps:")
print((summary["resolved_at_versions"] > 1).sum())

print("\nIncidents with changing priority:")
print((summary["priority_versions"] > 1).sum())

print("\nIncidents with changing category:")
print((summary["category_versions"] > 1).sum())

print("\nIncidents with reopen_count > 0:")
print((summary["max_reopen_count"] > 0).sum())

print("\nIncidents with no resolution timestamp:")
print((summary["resolved_at_versions"] == 0).sum())

print("\nExample incident inconsistencies:")
display(
    summary[
        (summary["resolved_at_versions"] > 1)
        | (summary["opened_at_versions"] > 1)
    ].head(10)
)


Total unique incidents: 24918

Incidents with multiple opening timestamps:
0

Incidents with multiple resolution timestamps:
0

Incidents with changing priority:
383

Incidents with changing category:
1184

Incidents with reopen_count > 0:
275

Incidents with no resolution timestamp:
1556

Example incident inconsistencies:


,event_count,opened_at_versions,resolved_at_versions,priority_versions,category_versions,max_reopen_count
number,,,,,,


In [7]:

inspect_ids = [
    "INC0000045",
    "INC0000047",
    "INC0000130"
]

columns = [
    "number",
    "incident_state",
    "priority",
    "opened_at",
    "sys_updated_at",
    "resolved_at",
    "reopen_count",
    "sys_mod_count"
]

for incident_id in inspect_ids:
    print(f"\n===== {incident_id} =====")

    incident_rows = df[
        df["number"] == incident_id
    ].sort_values("sys_updated_at_parsed")

    display(incident_rows[columns])



===== INC0000045 =====


,number,incident_state,priority,opened_at,sys_updated_at,resolved_at,reopen_count,sys_mod_count
0,INC0000045,New,3 - Moderate,29/2/2016 01:16,29/2/2016 01:23,29/2/2016 11:29,0,0
1,INC0000045,Resolved,3 - Moderate,29/2/2016 01:16,29/2/2016 08:53,29/2/2016 11:29,0,2
2,INC0000045,Resolved,3 - Moderate,29/2/2016 01:16,29/2/2016 11:29,29/2/2016 11:29,0,3
3,INC0000045,Closed,3 - Moderate,29/2/2016 01:16,5/3/2016 12:00,29/2/2016 11:29,0,4



===== INC0000047 =====


,number,incident_state,priority,opened_at,sys_updated_at,resolved_at,reopen_count,sys_mod_count
4,INC0000047,New,3 - Moderate,29/2/2016 04:40,29/2/2016 04:57,1/3/2016 09:52,0,0
5,INC0000047,Active,3 - Moderate,29/2/2016 04:40,29/2/2016 05:30,1/3/2016 09:52,0,1
6,INC0000047,Active,3 - Moderate,29/2/2016 04:40,29/2/2016 05:33,1/3/2016 09:52,0,2
7,INC0000047,Active,3 - Moderate,29/2/2016 04:40,29/2/2016 11:31,1/3/2016 09:52,0,3
8,INC0000047,Active,3 - Moderate,29/2/2016 04:40,29/2/2016 11:32,1/3/2016 09:52,0,4
9,INC0000047,Active,3 - Moderate,29/2/2016 04:40,1/3/2016 09:14,1/3/2016 09:52,0,5
10,INC0000047,Awaiting User Info,3 - Moderate,29/2/2016 04:40,1/3/2016 09:14,1/3/2016 09:52,0,6
11,INC0000047,Resolved,3 - Moderate,29/2/2016 04:40,1/3/2016 09:52,1/3/2016 09:52,0,7
12,INC0000047,Closed,3 - Moderate,29/2/2016 04:40,6/3/2016 10:00,1/3/2016 09:52,0,8



===== INC0000130 =====


,number,incident_state,priority,opened_at,sys_updated_at,resolved_at,reopen_count,sys_mod_count
328,INC0000130,New,3 - Moderate,29/2/2016 09:41,29/2/2016 07:27,3/3/2016 10:16,0,2
329,INC0000130,New,3 - Moderate,29/2/2016 09:41,29/2/2016 09:59,3/3/2016 10:16,0,1
330,INC0000130,Active,3 - Moderate,29/2/2016 09:41,1/3/2016 14:13,3/3/2016 10:16,0,2
331,INC0000130,Active,3 - Moderate,29/2/2016 09:41,3/3/2016 10:16,3/3/2016 10:16,0,3
332,INC0000130,Resolved,3 - Moderate,29/2/2016 09:41,3/3/2016 10:16,3/3/2016 10:16,0,4
333,INC0000130,Closed,3 - Moderate,29/2/2016 09:41,8/3/2016 11:00,3/3/2016 10:16,0,5


In [8]:

import pandas as pd

work = df.copy()

# Use parsed timestamps from previous cells
work = work.sort_values(
    ["number", "sys_updated_at_parsed"],
    kind="stable"
)

# One earliest available event per incident
first_event = (
    work.groupby("number", sort=False)
    .head(1)
    .set_index("number")
)

# Incident consistency summary from Cell 5
audit = summary.copy()

audit["has_reopen"] = audit["max_reopen_count"] > 0
audit["priority_changed"] = audit["priority_versions"] > 1
audit["category_changed"] = audit["category_versions"] > 1
audit["missing_resolution"] = audit["resolved_at_versions"] == 0

# Detect invalid update chronology at incident level
bad_chronology = work.loc[
    work["sys_updated_at_parsed"]
    < work["opened_at_parsed"],
    "number"
].unique()

audit["bad_chronology"] = audit.index.isin(
    bad_chronology
)

# Conservative initial candidate policy
audit["candidate"] = ~(
    audit["has_reopen"]
    | audit["priority_changed"]
    | audit["category_changed"]
    | audit["missing_resolution"]
    | audit["bad_chronology"]
)

# First-event feature availability
audit["first_priority"] = first_event["priority"]
audit["first_category"] = first_event["category"]
audit["first_state"] = first_event["incident_state"]

print("Total incidents:", len(audit))
print("Initial conservative candidates:",
      int(audit["candidate"].sum()))

print("\nExclusion flags:")
for col in [
    "has_reopen",
    "priority_changed",
    "category_changed",
    "missing_resolution",
    "bad_chronology"
]:
    print(col, int(audit[col].sum()))

print("\nFirst-event priority distribution:")
print(
    audit.loc[audit["candidate"], "first_priority"]
    .value_counts(dropna=False)
)

print("\nFirst-event state distribution:")
print(
    audit.loc[audit["candidate"], "first_state"]
    .value_counts(dropna=False)
)

print("\nPotentially missing first categories:")
print(
    audit.loc[audit["candidate"], "first_category"]
    .isin(["", "?", "NULL", "null"])
    .sum()
)


Total incidents: 24918
Initial conservative candidates: 21597

Exclusion flags:
has_reopen 275
priority_changed 383
category_changed 1184
missing_resolution 1556
bad_chronology 5

First-event priority distribution:
first_priority
3 - Moderate    20614
4 - Low           600
2 - High          268
1 - Critical      115
Name: count, dtype: int64

First-event state distribution:
first_state
New                   15086
Active                 6051
Resolved                274
Awaiting User Info      180
Awaiting Problem          6
Name: count, dtype: int64

Potentially missing first categories:
36


In [9]:

import pandas as pd

# Proposed MVP SLA targets in hours
sla_hours = {
    "1 - Critical": 4,
    "2 - High": 8,
    "3 - Moderate": 24,
    "4 - Low": 72
}

# Keep event ordering stable
events = df.sort_values(
    ["number", "sys_updated_at_parsed"],
    kind="stable"
)

# First observed event per incident
first = (
    events.groupby("number", sort=False)
    .head(1)
    .set_index("number")
)

# Construct a read-only validation table
check = audit[["candidate"]].copy()

check["priority"] = first["priority"]
check["first_state"] = first["incident_state"]
check["opened_at"] = first["opened_at_parsed"]
check["snapshot_at"] = first["sys_updated_at_parsed"]
check["resolved_at"] = first["resolved_at_parsed"]

check["sla_hours"] = check["priority"].map(sla_hours)

check["deadline"] = (
    check["opened_at"]
    + pd.to_timedelta(check["sla_hours"], unit="h")
)

# Conservative eligible states
open_states = ["New", "Active"]

check["valid_state"] = (
    check["first_state"].isin(open_states)
)

check["valid_timestamps"] = (
    check["opened_at"].notna()
    & check["snapshot_at"].notna()
    & check["resolved_at"].notna()
    & check["deadline"].notna()
)

check["snapshot_in_window"] = (
    (check["snapshot_at"] >= check["opened_at"])
    & (check["snapshot_at"] < check["deadline"])
)

check["resolution_after_snapshot"] = (
    check["resolved_at"] > check["snapshot_at"]
)

check["eligible"] = (
    check["candidate"]
    & check["valid_state"]
    & check["valid_timestamps"]
    & check["snapshot_in_window"]
    & check["resolution_after_snapshot"]
)

# Construct proposed breach labels for eligible incidents
eligible = check.loc[check["eligible"]].copy()

eligible["y_breach"] = (
    eligible["resolved_at"] > eligible["deadline"]
).astype(int)

eligible["ticket_age_hours"] = (
    eligible["snapshot_at"] - eligible["opened_at"]
).dt.total_seconds() / 3600

eligible["remaining_sla_hours"] = (
    eligible["deadline"] - eligible["snapshot_at"]
).dt.total_seconds() / 3600

print("Total incidents:", len(check))
print("Initial candidates:", int(check["candidate"].sum()))

print("\nEligibility checks within candidates:")
candidates = check.loc[check["candidate"]]

for col in [
    "valid_state",
    "valid_timestamps",
    "snapshot_in_window",
    "resolution_after_snapshot"
]:
    print(col, int(candidates[col].sum()))

print("\nEligible incidents:", len(eligible))

print("\nProvisional SLA labels:")
print(eligible["y_breach"].value_counts(dropna=False))

print("\nProvisional breach rate by priority:")
print(
    eligible.groupby("priority")["y_breach"]
    .agg(["count", "sum", "mean"])
)

print("\nTicket age statistics:")
print(eligible["ticket_age_hours"].describe())

print("\nRemaining SLA hours statistics:")
print(eligible["remaining_sla_hours"].describe())


Total incidents: 24918
Initial candidates: 21597

Eligibility checks within candidates:
valid_state 21137
valid_timestamps 21597
snapshot_in_window 21596
resolution_after_snapshot 21129

Eligible incidents: 20940

Provisional SLA labels:
y_breach
0    11256
1     9684
Name: count, dtype: int64

Provisional breach rate by priority:
              count   sum      mean
priority                           
1 - Critical    113    88  0.778761
2 - High        267   187  0.700375
3 - Moderate  20003  9251  0.462481
4 - Low         557   158  0.283662

Ticket age statistics:
count    20940.000000
mean         0.087810
std          0.230003
min          0.000000
25%          0.000000
50%          0.016667
75%          0.100000
max         19.100000
Name: ticket_age_hours, dtype: float64

Remaining SLA hours statistics:
count    20940.000000
mean        24.877042
std          8.116577
min          3.283333
25%         23.900000
50%         23.983333
75%         24.000000
max         72.000000
Nam

In [10]:

import pandas as pd

# Reuse df, check, eligible and summary from earlier cells.
events = df.sort_values(
    ["number", "sys_updated_at_parsed"],
    kind="stable"
)

# Last observed event for each incident
last = (
    events.groupby("number", sort=False)
    .tail(1)
    .set_index("number")
)

# First observed event for each incident
first = (
    events.groupby("number", sort=False)
    .head(1)
    .set_index("number")
)

audit9 = eligible.copy()

audit9["last_state"] = last["incident_state"]
audit9["last_updated_at"] = last["sys_updated_at_parsed"]
audit9["last_resolved_at"] = last["resolved_at_parsed"]

print("Eligible incidents:", len(audit9))

print("\nLast observed state distribution:")
print(audit9["last_state"].value_counts(dropna=False))

# Final outcome consistency
print("\nFinal state not Resolved or Closed:")
bad_final_state = ~audit9["last_state"].isin(
    ["Resolved", "Closed"]
)
print(int(bad_final_state.sum()))

# Resolution timestamp later than last observed update
print("\nResolution later than last recorded update:")
future_final_resolution = (
    audit9["last_resolved_at"]
    > audit9["last_updated_at"]
)
print(int(future_final_resolution.sum()))

# States that suggest resolution before the recorded time
suspicious_state = (
    events["incident_state"].isin(["Resolved", "Closed"])
    & events["resolved_at_parsed"].notna()
    & events["sys_updated_at_parsed"].notna()
    & (
        events["sys_updated_at_parsed"]
        < events["resolved_at_parsed"]
    )
)

print("\nResolved/Closed event rows before resolved_at:")
print(int(suspicious_state.sum()))

print("Unique incidents affected:")
print(events.loc[suspicious_state, "number"].nunique())

# Assess selection caused by future changes
potentially_biased = (
    (summary["priority_versions"] > 1)
    | (summary["category_versions"] > 1)
)

print("\nIncidents excluded due to future priority/category changes:")
print(int(potentially_biased.sum()))

# Check observed resolution consistency
print("\nFirst vs last resolution timestamps differ:")
different_resolution = (
    first["resolved_at_parsed"].notna()
    & last["resolved_at_parsed"].notna()
    & (
        first["resolved_at_parsed"]
        != last["resolved_at_parsed"]
    )
)
print(int(different_resolution.sum()))

# Audit timestamp ties
print("\nIncidents with repeated update timestamps:")
repeated_updates = events.duplicated(
    ["number", "sys_updated_at_parsed"],
    keep=False
)
print(events.loc[repeated_updates, "number"].nunique())

print("\nExample suspicious state transitions:")
display(
    events.loc[
        suspicious_state,
        [
            "number",
            "incident_state",
            "sys_updated_at",
            "resolved_at"
        ]
    ].head(10)
)


Eligible incidents: 20940

Last observed state distribution:
last_state
Closed    20940
Name: count, dtype: int64

Final state not Resolved or Closed:
0

Resolution later than last recorded update:
0

Resolved/Closed event rows before resolved_at:
742
Unique incidents affected:
558

Incidents excluded due to future priority/category changes:
1515

First vs last resolution timestamps differ:
0

Incidents with repeated update timestamps:
9318

Example suspicious state transitions:


,number,incident_state,sys_updated_at,resolved_at
1,INC0000045,Resolved,29/2/2016 08:53,29/2/2016 11:29
216,INC0000102,Resolved,14/3/2016 08:13,17/3/2016 16:28
354,INC0000135,Resolved,29/2/2016 15:27,29/2/2016 16:17
420,INC0000159,Resolved,2/3/2016 09:01,2/3/2016 10:09
579,INC0000218,Resolved,29/2/2016 12:08,29/2/2016 13:36
661,INC0000250,Resolved,12/3/2016 21:29,12/3/2016 21:30
833,INC0000294,Resolved,8/3/2016 11:30,14/3/2016 08:25
844,INC0000294,Resolved,14/3/2016 08:10,14/3/2016 08:25
1107,INC0000370,Resolved,1/3/2016 08:25,1/3/2016 08:31
1253,INC0000403,Resolved,7/3/2016 10:03,7/3/2016 16:33


In [11]:

import pandas as pd

# Preserve stable ordering
events = df.sort_values(
    ["number", "sys_updated_at_parsed"],
    kind="stable"
)

first = (
    events.groupby("number", sort=False)
    .head(1)
    .set_index("number")
)

last = (
    events.groupby("number", sort=False)
    .tail(1)
    .set_index("number")
)

sla_hours = {
    "1 - Critical": 4,
    "2 - High": 8,
    "3 - Moderate": 24,
    "4 - Low": 72
}

# Compare against the previous screening cohort
review = pd.DataFrame(index=first.index)

review["priority"] = first["priority"]
review["category"] = first["category"]
review["state"] = first["incident_state"]
review["opened_at"] = first["opened_at_parsed"]
review["snapshot_at"] = first["sys_updated_at_parsed"]

# Outcome information is for audit and labeling only
review["resolved_at"] = last["resolved_at_parsed"]
review["final_state"] = last["incident_state"]

review["sla_hours"] = review["priority"].map(sla_hours)

review["deadline"] = (
    review["opened_at"]
    + pd.to_timedelta(review["sla_hours"], unit="h")
)

# Keep outcome-based exclusions explicit
review["no_reopen"] = (
    summary["max_reopen_count"].fillna(0) == 0
)

review["valid_final_state"] = (
    review["final_state"].isin(["Resolved", "Closed"])
)

review["valid_first_state"] = (
    review["state"].isin(["New", "Active"])
)

review["valid_times"] = (
    review["opened_at"].notna()
    & review["snapshot_at"].notna()
    & review["resolved_at"].notna()
    & review["deadline"].notna()
    & (review["snapshot_at"] >= review["opened_at"])
    & (review["snapshot_at"] < review["deadline"])
    & (review["resolved_at"] > review["snapshot_at"])
)

review["eligible_v2"] = (
    review["no_reopen"]
    & review["valid_final_state"]
    & review["valid_first_state"]
    & review["valid_times"]
)

# Comparison only; NOT final training approval
review["previous_candidate"] = audit["candidate"]

print("Previous eligible count:", len(eligible))

print("Revised provisional count:",
      int(review["eligible_v2"].sum()))

print("Newly included incidents:",
      int(
          (
              review["eligible_v2"]
              & ~review["previous_candidate"]
          ).sum()
      ))

print("\nRevised cohort by priority:")
print(
    review.loc[
        review["eligible_v2"], "priority"
    ].value_counts()
)

print("\nChanges in priority/category:")
changed = (
    (summary["priority_versions"] > 1)
    | (summary["category_versions"] > 1)
)

print("Revised cohort with historical changes:",
      int(
          (
              review["eligible_v2"] & changed
          ).sum()
      ))

print("\nRevised provisional labels:")
revised = review.loc[review["eligible_v2"]].copy()

revised["y_breach"] = (
    revised["resolved_at"]
    > revised["deadline"]
).astype(int)

print(revised["y_breach"].value_counts())

print("\nBreach rate by priority:")
print(
    revised.groupby("priority")["y_breach"]
    .agg(["count", "sum", "mean"])
)


Previous eligible count: 20940
Revised provisional count: 22413
Newly included incidents: 1473

Revised cohort by priority:
priority
3 - Moderate    21328
4 - Low           594
2 - High          309
1 - Critical      182
Name: count, dtype: int64

Changes in priority/category:
Revised cohort with historical changes: 1473

Revised provisional labels:
y_breach
0    11545
1    10868
Name: count, dtype: int64

Breach rate by priority:
              count    sum      mean
priority                            
1 - Critical    182    149  0.818681
2 - High        309    221  0.715210
3 - Moderate  21328  10313  0.483543
4 - Low         594    185  0.311448


In [12]:

import pandas as pd

events = df.sort_values(
    ["number", "sys_updated_at_parsed"],
    kind="stable"
).copy()

# Identify the earliest timestamp for each incident
first_timestamp = events.groupby("number")[
    "sys_updated_at_parsed"
].transform("min")

earliest_rows = events.loc[
    events["sys_updated_at_parsed"] == first_timestamp
].copy()

# Count earliest rows per incident
tie_counts = earliest_rows.groupby("number").size()

tied_ids = tie_counts[tie_counts > 1].index

print("Incidents with tied earliest timestamps:",
      len(tied_ids))

# Check feature conflicts among tied first rows
tie_groups = earliest_rows[
    earliest_rows["number"].isin(tied_ids)
].groupby("number")

conflicts = pd.DataFrame({
    "priority_conflict":
        tie_groups["priority"].nunique() > 1,
    "category_conflict":
        tie_groups["category"].nunique() > 1,
    "state_conflict":
        tie_groups["incident_state"].nunique() > 1
})

conflicts["any_conflict"] = conflicts.any(axis=1)

print("\nConflicting earliest observations:")
print(conflicts.sum())

# How many revised candidates are affected?
revised_ids = revised.index

affected = revised_ids.intersection(
    conflicts.index[conflicts["any_conflict"]]
)

print("\nRevised candidates with ambiguous first event:",
      len(affected))

# Examine examples without altering the dataset
print("\nExamples:")
display(
    earliest_rows[
        earliest_rows["number"].isin(affected[:5])
    ][[
        "number",
        "sys_updated_at",
        "incident_state",
        "priority",
        "category",
        "sys_mod_count"
    ]].head(25)
)


Incidents with tied earliest timestamps: 2212

Conflicting earliest observations:
priority_conflict     15
category_conflict      7
state_conflict       829
any_conflict         848
dtype: int64

Revised candidates with ambiguous first event: 625

Examples:


,number,sys_updated_at,incident_state,priority,category,sys_mod_count
509,INC0000193,29/2/2016 11:27,New,3 - Moderate,Category 37,0
510,INC0000193,29/2/2016 11:27,Active,3 - Moderate,Category 37,1
675,INC0000259,29/2/2016 13:15,New,3 - Moderate,Category 9,0
676,INC0000259,29/2/2016 13:15,Active,3 - Moderate,Category 9,1
1144,INC0000378,29/2/2016 16:28,Active,3 - Moderate,Category 45,0
1145,INC0000378,29/2/2016 16:28,Awaiting User Info,3 - Moderate,Category 45,1
2570,INC0000749,1/3/2016 14:09,New,3 - Moderate,Category 42,0
2571,INC0000749,1/3/2016 14:09,Active,3 - Moderate,Category 42,1
3769,INC0000991,2/3/2016 08:18,New,3 - Moderate,Category 24,0
3770,INC0000991,2/3/2016 08:18,New,2 - High,Category 24,1


In [13]:

import pandas as pd

events = df.sort_values(
    ["number", "sys_updated_at_parsed"],
    kind="stable"
)

first_time = events.groupby("number")[
    "sys_updated_at_parsed"
].transform("min")

early = events.loc[
    events["sys_updated_at_parsed"] == first_time
].copy()

open_states = {"New", "Active"}

def classify_tie(group):
    priorities = set(group["priority"].dropna())
    categories = set(group["category"].dropna())
    states = set(group["incident_state"].dropna())

    # Missing tokens should not be treated
    # as meaningful categories.
    priorities.discard("?")
    categories.discard("?")

    has_feature_conflict = (
        len(priorities) > 1
        or len(categories) > 1
    )

    all_open = bool(states) and states.issubset(
        open_states
    )

    return pd.Series({
        "event_count": len(group),
        "priority_conflict": len(priorities) > 1,
        "category_conflict": len(categories) > 1,
        "all_states_open": all_open,
        "safe_initial_observation": (
            not has_feature_conflict and all_open
        )
    })

tie_audit = early.groupby("number").apply(
    classify_tie
)

# Restrict reporting to the revised provisional cohort
cohort_audit = tie_audit.loc[
    tie_audit.index.intersection(revised.index)
]

print("Revised provisional cohort:", len(revised))

print("\nSafe initial observations:",
      int(cohort_audit["safe_initial_observation"].sum()))

print("\nAmbiguous initial observations:",
      int((~cohort_audit["safe_initial_observation"]).sum()))

print("\nPriority conflicts:",
      int(cohort_audit["priority_conflict"].sum()))

print("\nCategory conflicts:",
      int(cohort_audit["category_conflict"].sum()))

print("\nEarliest states not all New/Active:",
      int((~cohort_audit["all_states_open"]).sum()))

print("\nAmbiguous examples:")
ambiguous_ids = cohort_audit.index[
    ~cohort_audit["safe_initial_observation"]
]

display(
    early[early["number"].isin(ambiguous_ids[:10])][
        [
            "number",
            "sys_updated_at",
            "incident_state",
            "priority",
            "category"
        ]
    ].head(30)
)


Revised provisional cohort: 22413

Safe initial observations: 22320

Ambiguous initial observations: 93

Priority conflicts: 15

Category conflicts: 7

Earliest states not all New/Active: 71

Ambiguous examples:


/tmp/ipykernel_785/4287248208.py:47: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tie_audit = early.groupby("number").apply(


,number,sys_updated_at,incident_state,priority,category
1144,INC0000378,29/2/2016 16:28,Active,3 - Moderate,Category 45
1145,INC0000378,29/2/2016 16:28,Awaiting User Info,3 - Moderate,Category 45
3769,INC0000991,2/3/2016 08:18,New,3 - Moderate,Category 24
3770,INC0000991,2/3/2016 08:18,New,2 - High,Category 24
5054,INC0001248,2/3/2016 12:10,Active,3 - Moderate,Category 45
5055,INC0001248,2/3/2016 12:10,Awaiting User Info,3 - Moderate,Category 45
5148,INC0001263,2/3/2016 12:26,Active,3 - Moderate,Category 44
5149,INC0001263,2/3/2016 12:26,Awaiting User Info,3 - Moderate,Category 44
5150,INC0001263,2/3/2016 12:26,Awaiting User Info,3 - Moderate,Category 44
5663,INC0001358,2/3/2016 14:29,Active,3 - Moderate,Category 45


In [14]:

from pathlib import Path
import pandas as pd

# Choose incidents passing the consistency checks
safe_ids = cohort_audit.index[
    cohort_audit["safe_initial_observation"]
]

safe_ids = revised.index.intersection(safe_ids)

# Use the first observed event to obtain
# prediction-time feature values
events = df.sort_values(
    ["number", "sys_updated_at_parsed"],
    kind="stable"
)

first = (
    events.groupby("number", sort=False)
    .head(1)
    .set_index("number")
)

# Reuse outcome and deadline information
# from the provisional labeling audit
ml_data = revised.loc[safe_ids].copy()

# Priority/category are consistent across tied
# earliest rows for the selected incidents
ml_data["priority"] = first.loc[
    safe_ids, "priority"
]

ml_data["category"] = first.loc[
    safe_ids, "category"
].replace(
    {"?": "Unknown", "": "Unknown"}
)

# Prediction-time features
ml_data["ticket_age_hours"] = (
    ml_data["snapshot_at"]
    - ml_data["opened_at"]
).dt.total_seconds() / 3600

ml_data["remaining_sla_hours"] = (
    ml_data["deadline"]
    - ml_data["snapshot_at"]
).dt.total_seconds() / 3600

ml_data["created_hour"] = (
    ml_data["opened_at"].dt.hour
)

ml_data["created_weekday"] = (
    ml_data["opened_at"].dt.dayofweek
)

# Preserve one row per incident
ml_data["ticket_id"] = ml_data.index

# Keep metadata for splitting and auditing
output_columns = [
    "ticket_id",
    "opened_at",
    "snapshot_at",
    "resolved_at",
    "deadline",
    "priority",
    "category",
    "ticket_age_hours",
    "remaining_sla_hours",
    "created_hour",
    "created_weekday",
    "y_breach"
]

ml_data = ml_data[output_columns].copy()

# Validate candidate structure
assert ml_data["ticket_id"].is_unique
assert not ml_data["y_breach"].isna().any()
assert ml_data["y_breach"].isin([0, 1]).all()
assert (ml_data["ticket_age_hours"] >= 0).all()
assert (ml_data["remaining_sla_hours"] > 0).all()

print("Provisional dataset shape:", ml_data.shape)

print("\nSLA label distribution:")
print(ml_data["y_breach"].value_counts())

print("\nPriority distribution:")
print(ml_data["priority"].value_counts())

print("\nMissing values:")
print(ml_data.isna().sum())

display(ml_data.head())

# Save a research artifact, NOT yet a final
# approved training dataset
output_dir = Path("/content/sla_guardian_data")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "provisional_ml_dataset.csv"

ml_data.to_csv(output_path, index=False)

print("\nSaved:", output_path)


Provisional dataset shape: (22320, 12)

SLA label distribution:
y_breach
0    11514
1    10806
Name: count, dtype: int64

Priority distribution:
priority
3 - Moderate    21239
4 - Low           590
2 - High          309
1 - Critical      182
Name: count, dtype: int64

Missing values:
ticket_id              0
opened_at              0
snapshot_at            0
resolved_at            0
deadline               0
priority               0
category               0
ticket_age_hours       0
remaining_sla_hours    0
created_hour           0
created_weekday        0
y_breach               0
dtype: int64


,ticket_id,opened_at,snapshot_at,resolved_at,deadline,priority,category,ticket_age_hours,remaining_sla_hours,created_hour,created_weekday,y_breach
number,,,,,,,,,,,,
INC0000045,INC0000045,2016-02-29 01:16:00,2016-02-29 01:23:00,2016-02-29 11:29:00,2016-03-01 01:16:00,3 - Moderate,Category 55,0.116667,23.883333,1,0,0
INC0000047,INC0000047,2016-02-29 04:40:00,2016-02-29 04:57:00,2016-03-01 09:52:00,2016-03-01 04:40:00,3 - Moderate,Category 40,0.283333,23.716667,4,0,1
INC0000057,INC0000057,2016-02-29 06:10:00,2016-02-29 06:26:00,2016-03-01 02:55:00,2016-03-01 06:10:00,3 - Moderate,Category 20,0.266667,23.733333,6,0,0
INC0000060,INC0000060,2016-02-29 06:38:00,2016-02-29 06:42:00,2016-03-02 12:06:00,2016-03-01 06:38:00,3 - Moderate,Category 9,0.066667,23.933333,6,0,1
INC0000062,INC0000062,2016-02-29 06:58:00,2016-02-29 07:26:00,2016-02-29 15:51:00,2016-03-01 06:58:00,3 - Moderate,Category 53,0.466667,23.533333,6,0,0



Saved: /content/sla_guardian_data/provisional_ml_dataset.csv


In [15]:

import pandas as pd

# Use the completed Cell 13 dataset
data = ml_data.copy()

# Chronological ordering using prediction time
data = data.sort_values(
    ["snapshot_at", "ticket_id"],
    kind="stable"
).reset_index(drop=True)

assert data["ticket_id"].is_unique

n = len(data)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = data.iloc[:train_end].copy()
val = data.iloc[train_end:val_end].copy()
test = data.iloc[val_end:].copy()

# Dates when the next period starts
validation_start = val["snapshot_at"].min()
test_start = test["snapshot_at"].min()

# Audit outcome availability before the next period
train_mature = (
    train["resolved_at"] <= validation_start
)

val_mature = (
    val["resolved_at"] <= test_start
)

print("Original split sizes:")
print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))

print("\nBoundary dates:")
print("Validation starts:", validation_start)
print("Test starts:", test_start)

print("\nOutcome maturity audit:")
print("Train outcomes available by validation start:",
      int(train_mature.sum()))
print("Train outcomes not yet available:",
      int((~train_mature).sum()))

print("Validation outcomes available by test start:",
      int(val_mature.sum()))
print("Validation outcomes not yet available:",
      int((~val_mature).sum()))

# Produce conservative, outcome-matured partitions
train_ready = train.loc[train_mature].copy()
val_ready = val.loc[val_mature].copy()
test_ready = test.copy()

# Incident isolation checks
assert set(train_ready.ticket_id).isdisjoint(
    set(val_ready.ticket_id)
)
assert set(train_ready.ticket_id).isdisjoint(
    set(test_ready.ticket_id)
)
assert set(val_ready.ticket_id).isdisjoint(
    set(test_ready.ticket_id)
)

feature_cols = [
    "priority",
    "category",
    "ticket_age_hours",
    "remaining_sla_hours",
    "created_hour",
    "created_weekday"
]

print("\nConservative partitions:")
for name, part in [
    ("Train", train_ready),
    ("Validation", val_ready),
    ("Test", test_ready)
]:
    print(f"\n{name}: {len(part)}")
    print(part["y_breach"].value_counts())
    print("Breach rate:",
          round(part["y_breach"].mean(), 4))

print("\nApproved model inputs:")
print(feature_cols)

print("\nSplit isolation: PASS")


Original split sizes:
Train: 15623
Validation: 3349
Test: 3348

Boundary dates:
Validation starts: 2016-05-03 05:59:00
Test starts: 2016-05-16 17:33:00

Outcome maturity audit:
Train outcomes available by validation start: 14435
Train outcomes not yet available: 1188
Validation outcomes available by test start: 2671
Validation outcomes not yet available: 678

Conservative partitions:

Train: 14435
y_breach
0    7317
1    7118
Name: count, dtype: int64
Breach rate: 0.4931

Validation: 2671
y_breach
0    2030
1     641
Name: count, dtype: int64
Breach rate: 0.24

Test: 3348
y_breach
0    2111
1    1237
Name: count, dtype: int64
Breach rate: 0.3695

Approved model inputs:
['priority', 'category', 'ticket_age_hours', 'remaining_sla_hours', 'created_hour', 'created_weekday']

Split isolation: PASS


In [16]:

import pandas as pd

def maturity_report(part, cutoff, name):
    temp = part.copy()

    temp["outcome_available"] = (
        temp["resolved_at"] <= cutoff
    )

    temp["resolution_hours"] = (
        temp["resolved_at"] - temp["opened_at"]
    ).dt.total_seconds() / 3600

    print(f"\n===== {name} =====")
    print("Total records:", len(temp))
    print("Available outcomes:",
          int(temp["outcome_available"].sum()))
    print("Unavailable outcomes:",
          int((~temp["outcome_available"]).sum()))

    print("\nBreach rate by outcome availability:")
    print(
        temp.groupby("outcome_available")["y_breach"]
        .agg(["count", "sum", "mean"])
    )

    print("\nResolution duration by availability:")
    print(
        temp.groupby("outcome_available")[
            "resolution_hours"
        ].describe()
    )

    print("\nPriority distribution:")
    print(
        pd.crosstab(
            temp["priority"],
            temp["outcome_available"]
        )
    )

maturity_report(
    train,
    validation_start,
    "TRAINING"
)

maturity_report(
    val,
    test_start,
    "VALIDATION"
)

print("\n===== TEST OBSERVATION WINDOW =====")
print("Earliest test snapshot:", test["snapshot_at"].min())
print("Latest test snapshot:", test["snapshot_at"].max())
print("Latest test resolution:", test["resolved_at"].max())

print("\n===== FULL PARTITION LABEL RATES =====")
for name, part in [
    ("Train original", train),
    ("Validation original", val),
    ("Test original", test)
]:
    print(
        name,
        "records:", len(part),
        "breach rate:",
        round(part["y_breach"].mean(), 4)
    )




===== TRAINING =====
Total records: 15623
Available outcomes: 14435
Unavailable outcomes: 1188

Breach rate by outcome availability:
                   count   sum      mean
outcome_available                       
False               1188  1153  0.970539
True               14435  7118  0.493107

Resolution duration by availability:
                     count         mean          std        min         25%  \
outcome_available                                                             
False               1188.0  1321.193042  1522.496996  11.516667  220.104167   
True               14435.0    98.837396   168.494187   0.016667    0.566667   

                          50%          75%          max  
outcome_available                                        
False              717.775000  1879.145833  8070.166667  
True                23.416667   139.366667  1434.466667  

Priority distribution:
outcome_available  False  True 
priority                       
1 - Critical           9   

In [17]:

import pandas as pd

def add_label_availability(part):
    result = part.copy()

    # Breaches become known at the SLA deadline.
    # On-time outcomes become known at resolution.
    result["label_known_at"] = result["resolved_at"].where(
        result["y_breach"].eq(0),
        result["deadline"]
    )

    assert result["label_known_at"].notna().all()

    assert (
        result["label_known_at"] >= result["snapshot_at"]
    ).all()

    return result

train_v2 = add_label_availability(train)
val_v2 = add_label_availability(val)
test_v2 = add_label_availability(test)

# Outcome knowledge at each historical boundary
train_ready_v2 = train_v2.loc[
    train_v2["label_known_at"] <= validation_start
].copy()

val_ready_v2 = val_v2.loc[
    val_v2["label_known_at"] <= test_start
].copy()

print("===== CORRECTED OUTCOME MATURITY =====")

print("\nTraining:")
print("Original:", len(train_v2))
print("Previously retained:", len(train_ready))
print("Now label-known:", len(train_ready_v2))
print("Still unknown:",
      len(train_v2) - len(train_ready_v2))

print("\nValidation:")
print("Original:", len(val_v2))
print("Previously retained:", len(val_ready))
print("Now label-known:", len(val_ready_v2))
print("Still unknown:",
      len(val_v2) - len(val_ready_v2))

print("\n===== CORRECTED LABEL DISTRIBUTIONS =====")

for name, part in [
    ("Training label-known", train_ready_v2),
    ("Validation label-known", val_ready_v2),
    ("Original test", test_v2)
]:
    print(f"\n{name}: {len(part)}")
    print(part["y_breach"].value_counts())
    print("Breach rate:",
          round(part["y_breach"].mean(), 4))

# Reconfirm incident isolation
assert set(train_v2["ticket_id"]).isdisjoint(
    set(val_v2["ticket_id"])
)

assert set(train_v2["ticket_id"]).isdisjoint(
    set(test_v2["ticket_id"])
)

assert set(val_v2["ticket_id"]).isdisjoint(
    set(test_v2["ticket_id"])
)

print("\nIncident isolation: PASS")

print("\n===== TEST OUTCOME COVERAGE =====")
print("Latest test label-known time:",
      test_v2["label_known_at"].max())

print("Latest recorded source update:",
      df["sys_updated_at_parsed"].max())


===== CORRECTED OUTCOME MATURITY =====

Training:
Original: 15623
Previously retained: 14435
Now label-known: 15447
Still unknown: 176

Validation:
Original: 3349
Previously retained: 2671
Now label-known: 3221
Still unknown: 128

===== CORRECTED LABEL DISTRIBUTIONS =====

Training label-known: 15447
y_breach
1    8130
0    7317
Name: count, dtype: int64
Breach rate: 0.5263

Validation label-known: 3221
y_breach
0    2030
1    1191
Name: count, dtype: int64
Breach rate: 0.3698

Original test: 3348
y_breach
0    2111
1    1237
Name: count, dtype: int64
Breach rate: 0.3695

Incident isolation: PASS

===== TEST OUTCOME COVERAGE =====
Latest test label-known time: 2017-02-16 16:38:00
Latest recorded source update: 2017-02-18 15:00:00


In [18]:

from pathlib import Path
import pandas as pd
import json
import hashlib

output_dir = Path("/content/sla_guardian_data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Full chronological splits from Cell 14
partitions = {
    "train": train_v2,
    "validation": val_v2,
    "test": test_v2
}

# Save split assignments and historical labels
for name, part in partitions.items():
    path = output_dir / f"{name}_provisional.csv"
    part.to_csv(path, index=False)
    print(name, len(part), "->", path)

# Record the approved candidate features
feature_columns = [
    "priority",
    "category",
    "ticket_age_hours",
    "remaining_sla_hours",
    "created_hour",
    "created_weekday"
]

# Training labels known at validation start
train_available = train_v2[
    train_v2["label_known_at"] <= validation_start
].copy()

# Validation labels known at test start
validation_available = val_v2[
    val_v2["label_known_at"] <= test_start
].copy()

train_available.to_csv(
    output_dir / "train_cutoff_eligible.csv",
    index=False
)

validation_available.to_csv(
    output_dir / "validation_cutoff_eligible.csv",
    index=False
)

# SHA256 helper for reproducibility
def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

files = sorted(output_dir.glob("*.csv"))

manifest = {
    "project": "SLA Guardian AI",
    "status": "provisional_research_dataset",
    "source": "UCI Incident Management Process Enriched Event Log",
    "feature_columns": feature_columns,
    "label": "y_breach",
    "sla_policy_hours": {
        "critical": 4,
        "high": 8,
        "moderate": 24,
        "low": 72
    },
    "split_method": "chronological_by_snapshot_at",
    "validation_start": str(validation_start),
    "test_start": str(test_start),
    "source_timezone": "unverified",
    "limitations": [
        "Illustrative 24/7 SLA policy",
        "Earliest observed event used as feature snapshot",
        "Reopened incidents excluded using historical outcomes",
        "Historical event fields may contain backfilled values",
        "Same-minute update ambiguity remains a source limitation",
        "Outcome-maturity filtering creates a cutoff-dependent cohort"
    ],
    "counts": {
        "train_full": len(train_v2),
        "validation_full": len(val_v2),
        "test_full": len(test_v2),
        "train_cutoff_eligible": len(train_available),
        "validation_cutoff_eligible": len(validation_available)
    },
    "files": {
        f.name: file_sha256(f)
        for f in files
    }
}

manifest_path = output_dir / "dataset_manifest.json"

manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print("\nManifest saved:", manifest_path)
print(json.dumps(manifest["counts"], indent=2))

# Verify disjoint incident IDs
assert set(train_v2.ticket_id).isdisjoint(
    set(val_v2.ticket_id)
)
assert set(train_v2.ticket_id).isdisjoint(
    set(test_v2.ticket_id)
)
assert set(val_v2.ticket_id).isdisjoint(
    set(test_v2.ticket_id)
)

print("\nSplit isolation: PASS")
print("Files exported:", len(files))


train 15623 -> /content/sla_guardian_data/processed/train_provisional.csv
validation 3349 -> /content/sla_guardian_data/processed/validation_provisional.csv
test 3348 -> /content/sla_guardian_data/processed/test_provisional.csv

Manifest saved: /content/sla_guardian_data/processed/dataset_manifest.json
{
  "train_full": 15623,
  "validation_full": 3349,
  "test_full": 3348,
  "train_cutoff_eligible": 15447,
  "validation_cutoff_eligible": 3221
}

Split isolation: PASS
Files exported: 5
